In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install transformers==4.52.4  langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: langchain-core
    Found existing i

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt, max_length=1000, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs][0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [11]:
from langchain_core.language_models.llms import LLM
from typing import Any

class CustomHFLLM(LLM):
    def _call(self, prompt: str, stop: Any = None) -> str:
        return generate_text(prompt, max_length=500)
    
    @property
    def _llm_type(self) -> str:
        return "custom_huggingface"

llm = CustomHFLLM()

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

# Chain 1: Generate app idea
idea_prompt = PromptTemplate(
    input_variables=["topic"],
    template="Come up with a creative app idea related to {topic}. Keep it short."
)
idea_chain = LLMChain(llm=llm, prompt=idea_prompt)





# Chain 2: Extract features
feature_prompt = PromptTemplate(
    input_variables=["app_idea"],
    template="Given this app idea: {app_idea}, list its top 3 features in a bullet list."
)
feature_chain = LLMChain(llm=llm, prompt=feature_prompt)

# Chain 3: Create tagline
tagline_prompt = PromptTemplate(
    input_variables=["app_idea", "features"],
    template=(
        "Based on the app idea: {app_idea} and these features:\n{features},\n"
        "write a catchy tagline (1 sentence)."
    )
)

tagline_chain = LLMChain(llm=llm, prompt=tagline_prompt)

In [9]:
def run_all(topic: str):
    print(f"Topic: {topic}")
    
    app_idea = idea_chain.run(topic)
    print(f"\nApp Idea:\n{app_idea.strip()}")
    
    features = feature_chain.run(app_idea)
    print(f"\nFeatures:\n{features.strip()}")
    
    tagline = tagline_chain.run({
        "app_idea": app_idea,
        "features": features
    })
    print(f"\nTagline:\n{tagline.strip()}")

In [10]:
run_all("Football")

/tmp/ipykernel_55/1480299682.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  app_idea = idea_chain.run(topic)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Topic: Football


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



App Idea:
Come up with a creative app idea related to Football. Keep it short.

Title: "Pass Master"

Description: Pass Master is a mobile app that uses augmented reality (AR) technology to help football players improve their passing skills. Users can place virtual targets in their backyard or local park and practice passing to those targets, receiving instant feedback on accuracy, power, and trajectory. The app includes a variety of drills and challenges to help users progress and track their improvement over time. With leaderboards and social sharing, users can compete against friends and connect with other football players around the world.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Features:
Given this app idea: Come up with a creative app idea related to Football. Keep it short.

Title: "Pass Master"

Description: Pass Master is a mobile app that uses augmented reality (AR) technology to help football players improve their passing skills. Users can place virtual targets in their backyard or local park and practice passing to those targets, receiving instant feedback on accuracy, power, and trajectory. The app includes a variety of drills and challenges to help users progress and track their improvement over time. With leaderboards and social sharing, users can compete against friends and connect with other football players around the world., list its top 3 features in a bullet list.

* Augmented Reality Target Practice: Users can place virtual targets in their preferred location and practice passing to them, receiving instant feedback on accuracy, power, and trajectory.
* Customizable Drills and Challenges: The app includes a variety of drills and challenges to